In [1]:
import geopandas as gpd
import pandas as pd
from libpysal.weights import Queen
from esda.moran import Moran_Local

In [2]:
# --- Parameters ---
shapefile_path = "../../data/Local_Authority_Districts_(May_2025)_Boundaries_UK_BFC_(V2)/Local_Authority_Districts_(May_2025)_Boundaries_UK_BFC_(V2).shp"  # Change to your shapefile path
id_column = "LAD25CD"       # Column where first letter is E/W/S/N
name_column = "LAD25NM"   # Column with local authority names

# --- Load shapefile ---
gdf = gpd.read_file(shapefile_path)



In [4]:
gdf

,FID,LAD25CD,LAD25NM,LAD25NMW,BNG_E,BNG_N,LONG,LAT,Shape__Are,Shape__Len,GlobalID,geometry
0,1,E06000001,Hartlepool,None,447161,531473,-1.270174,54.676132,9.388700e+07,73982.408648,847f2c4b-a7cf-4c71-840c-0864853246d5,"MULTIPOLYGON (((450154.6 525938.201, 450140.09..."
1,2,E06000002,Middlesbrough,None,451141,516887,-1.210998,54.544679,5.388156e+07,44481.691242,f1925b75-6267-417d-a77a-05cdc4c6b1b3,"MULTIPOLYGON (((446854.7 517192.7, 446854.281 ..."
2,3,E06000003,Redcar and Cleveland,None,464330,519596,-1.006565,54.567520,2.451071e+08,97674.177085,36b1db27-3dfa-4ed6-8e81-36bf3abeeacc,"MULTIPOLYGON (((451747.397 520561.1, 451792.20..."
3,4,E06000004,Stockton-on-Tees,None,444940,518179,-1.306646,54.556876,2.049433e+08,123628.292301,22a6adf7-e812-4c09-89b1-6753ec35de93,"MULTIPOLYGON (((447177.704 517811.797, 447176...."
4,5,E06000005,Darlington,None,428029,515648,-1.568356,54.535345,1.974895e+08,107285.516031,309364b5-9b5c-4f9c-81f5-8a3a79699fd1,"POLYGON ((423496.602 524724.299, 423497.204 52..."
...,...,...,...,...,...,...,...,...,...,...,...,...
356,357,W06000020,Torfaen,Torfaen,327459,200480,-3.051019,51.698361,1.262410e+08,82554.336127,1c3eb529-b145-4eb7-84a6-b17ba581cbff,"POLYGON ((323898.201 211287.499, 324115.698 21..."
357,358,W06000021,Monmouthshire,Sir Fynwy,337812,209231,-2.902806,51.778278,8.503129e+08,224920.512820,640aefaf-22cf-49c5-a974-b43fce5f28c9,"MULTIPOLYGON (((345897.698 180999.599, 345884...."
358,359,W06000022,Newport,Casnewydd,337897,187432,-2.897690,51.582311,1.904380e+08,153232.897017,2cb099b7-0b64-4778-b41c-80d1cb832bae,"MULTIPOLYGON (((334186.001 192669.398, 334201...."
359,360,W06000023,Powys,Powys,302329,273254,-3.435318,52.348639,5.195310e+09,611011.547196,ef0eb86c-2311-4027-9e84-1a6520f67a65,"POLYGON ((322392.901 334017.198, 322378.002 33..."


In [5]:
gdf[['LAD25CD', 'LAD25NM']].to_csv("local_authority_districts.csv", index=False)

Centroids and rook contiguity neighbours

In [113]:
# Filter to only England & Wales
gdf = gdf[gdf[id_column].str[0].isin(["E", "W"])]

# Ensure consistent projection
gdf = gdf.to_crs(epsg=27700)  # British National Grid

# Compute centroids (in projected CRS)
gdf["centroid_x"] = gdf.geometry.centroid.x
gdf["centroid_y"] = gdf.geometry.centroid.y

# Build neighbor list (rook contiguity, optimized with spatial index)
rows = []
for idx, area in gdf.iterrows():
    # Use spatial index for speed
    possible_matches_index = list(gdf.sindex.intersection(area.geometry.bounds))
    possible_matches = gdf.iloc[possible_matches_index]

    # Find actual touching neighbors
    touching = possible_matches[possible_matches.geometry.touches(area.geometry)]

    # Append one row per neighbor
    for _, neighbor in touching.iterrows():
        rows.append({
            id_column: area[id_column],
            name_column: area[name_column],
            "neighbour_name": neighbor[name_column],
            "neighbour_id": neighbor[id_column],
            "centroid_x": area["centroid_x"],
            "centroid_y": area["centroid_y"]
        })

# Create DataFrame in long format
neighbors_df = pd.DataFrame(rows)



Load in Average prices, reduce to just LAs and England and Wales and remove Islands

In [114]:
la_list = gdf[['LAD25CD']]

hpi_raw = pd.read_csv(filepath_or_buffer="../../data/UK-HPI-full-file-2025-05.csv")
hpi_raw = hpi_raw[['Date', 'RegionName', 'AreaCode', 'AveragePrice']]
hpi_raw = hpi_raw[hpi_raw['AreaCode'].isin(la_list['LAD25CD'])]
hpi_raw['Date'] = pd.to_datetime(hpi_raw['Date'], format='%d/%m/%Y')

hpi_raw = hpi_raw[hpi_raw['AreaCode'] != 'W06000001'] # Exclude Isles of Anglesey as no neighbours
hpi_raw = hpi_raw[hpi_raw['AreaCode'] != 'E06000046'] # Exclude Isles of Wight as no neighbours

Calculate the average neighbours average value

In [115]:
la_neighbour_avg = []

for area in hpi_raw['AreaCode'].unique():
    area_neighbour_df = neighbors_df[neighbors_df['LAD25CD'] == area].copy()
    area_df = hpi_raw[hpi_raw['AreaCode'].isin(area_neighbour_df['neighbour_id'])].copy()

    neighbour_avg = area_df.groupby('Date').agg(
        AverageNeighbourPrice = ('AveragePrice', 'mean')
    ).reset_index()

    neighbour_avg['RegionName'] = area_neighbour_df['LAD25NM'].iloc[0]
    neighbour_avg['AreaCode'] = area_neighbour_df['LAD25CD'].iloc[0]

    # Store for concatenation
    la_neighbour_avg.append(neighbour_avg)

# Combine all area_code DataFrames into one
la_neighbour_avg = pd.concat(la_neighbour_avg, ignore_index=True)



In [116]:
max(hpi_raw['Date'])

df_month = hpi_raw[hpi_raw['Date'] == max(hpi_raw['Date'])]

In [117]:
gdf

,FID,LAD25CD,LAD25NM,LAD25NMW,BNG_E,BNG_N,LONG,LAT,Shape__Are,Shape__Len,GlobalID,geometry,centroid_x,centroid_y
0,1,E06000001,Hartlepool,None,447161,531473,-1.270174,54.676132,9.388700e+07,73982.408648,847f2c4b-a7cf-4c71-840c-0864853246d5,"MULTIPOLYGON (((450154.6 525938.201, 450140.09...",447873.372078,530735.890750
1,2,E06000002,Middlesbrough,None,451141,516887,-1.210998,54.544679,5.388156e+07,44481.691242,f1925b75-6267-417d-a77a-05cdc4c6b1b3,"MULTIPOLYGON (((446854.7 517192.7, 446854.281 ...",450415.357748,516581.889072
2,3,E06000003,Redcar and Cleveland,None,464330,519596,-1.006565,54.567520,2.451071e+08,97674.177085,36b1db27-3dfa-4ed6-8e81-36bf3abeeacc,"MULTIPOLYGON (((451747.397 520561.1, 451792.20...",463441.352192,517822.192022
3,4,E06000004,Stockton-on-Tees,None,444940,518179,-1.306646,54.556876,2.049433e+08,123628.292301,22a6adf7-e812-4c09-89b1-6753ec35de93,"MULTIPOLYGON (((447177.704 517811.797, 447176....",443273.979685,518687.338825
4,5,E06000005,Darlington,None,428029,515648,-1.568356,54.535345,1.974895e+08,107285.516031,309364b5-9b5c-4f9c-81f5-8a3a79699fd1,"POLYGON ((423496.602 524724.299, 423497.204 52...",429039.493772,517141.720170
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
356,357,W06000020,Torfaen,Torfaen,327459,200480,-3.051019,51.698361,1.262410e+08,82554.336127,1c3eb529-b145-4eb7-84a6-b17ba581cbff,"POLYGON ((323898.201 211287.499, 324115.698 21...",327292.493830,200507.820829
357,358,W06000021,Monmouthshire,Sir Fynwy,337812,209231,-2.902806,51.778278,8.503129e+08,224920.512820,640aefaf-22cf-49c5-a974-b43fce5f28c9,"MULTIPOLYGON (((345897.698 180999.599, 345884....",339910.997298,207836.640999
358,359,W06000022,Newport,Casnewydd,337897,187432,-2.897690,51.582311,1.904380e+08,153232.897017,2cb099b7-0b64-4778-b41c-80d1cb832bae,"MULTIPOLYGON (((334186.001 192669.398, 334201....",333166.626375,187164.374743
359,360,W06000023,Powys,Powys,302329,273254,-3.435318,52.348639,5.195310e+09,611011.547196,ef0eb86c-2311-4027-9e84-1a6520f67a65,"POLYGON ((322392.901 334017.198, 322378.002 33...",304005.227368,270451.604135


In [118]:
gdf = gdf[gdf['LAD25CD'] != 'W06000001'] # Exclude Isles of Anglesey as no neighbours
gdf = gdf[gdf['LAD25CD'] != 'E06000046']
gdf = gdf[gdf['LAD25CD'] != 'E06000053']

w_queen = Queen.from_dataframe(gdf)  
w_queen.transform = 'r'  # Row-standardize weights

C:\Users\slong\AppData\Local\Temp\ipykernel_13488\3123059998.py:5: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w_queen = Queen.from_dataframe(gdf)


In [119]:
print(w_queen.neighbors)  # dict: {LA_index: [neighbor_indices]}
print(w_queen.weights)    # dict: {LA_index: [weights]}

{0: [3, 43], 1: [59, 2, 3], 2: [1, 59], 3: [0, 1, 4, 59, 43], 4: [59, 3, 43], 5: [45, 6, 235, 236, 237], 6: [5, 230, 233, 234, 44, 45, 237], 7: [225, 226, 135, 137, 141, 142, 143], 8: [136, 145], 9: [10], 10: [240, 9, 59, 12, 13], 11: [154, 12, 159], 12: [240, 168, 10, 11, 159], 13: [10, 59], 14: [73, 66, 70], 15: [152, 146, 147, 148], 16: [158, 148, 150, 55], 17: [169, 170, 173, 167], 18: [101, 214, 311, 313, 46], 19: [184, 46, 183], 20: [184, 185, 182], 21: [48, 22, 23, 24, 60], 22: [24, 21, 23], 23: [60, 21, 22], 24: [48, 100, 21, 22, 103], 25: [78], 26: [78, 79], 27: [48, 177, 100], 28: [64, 55, 157, 158, 63], 29: [50, 119], 30: [96, 90], 31: [89, 276, 87], 32: [130, 132, 126, 127], 33: [197, 110, 37, 38], 34: [176, 177, 114, 35, 48, 38, 105], 35: [176, 34, 38], 36: [37, 196, 277, 54], 37: [33, 195, 196, 197, 38, 36, 54], 38: [33, 34, 35, 37, 105, 110, 176, 54], 39: [49, 50, 54, 55, 56], 40: [210, 211, 84, 206], 41: [115, 108, 111], 42: [114, 107], 43: [0, 258, 3, 4, 51, 245, 58, 5

In [ ]:
results = []

for month in hpi_raw['Date'].unique():
    month_data = hpi_raw[hpi_raw['Date'] == month]
    y = month_data['AveragePrice'].values
    
    moran_loc = Moran_Local(y, w_queen)
    
    month_results = pd.DataFrame({
        'local_I': moran_loc.Is,
        'p_value': moran_loc.p_sim,
        'quadrant': moran_loc.q
    }, index=month_data['AreaCode'].values)
    
    month_results['Date'] = month
    results.append(month_results)

local_moran_df = pd.concat(results)
local_moran_df.reset_index(inplace=True)

In [122]:
local_moran_df

,index,local_I,p_value,quadrant,Date
0,E07000223,-0.116671,0.064,2,1995-01-01
1,E07000032,0.343264,0.173,3,1995-01-01
2,E07000224,-0.056030,0.146,4,1995-01-01
3,E07000170,-0.327021,0.170,2,1995-01-01
4,E07000105,0.080845,0.157,1,1995-01-01
...,...,...,...,...,...
114970,W06000006,0.314289,0.175,3,2025-05-01
114971,E07000238,0.056677,0.013,3,2025-05-01
114972,E07000128,0.048554,0.485,3,2025-05-01
114973,E07000239,0.152093,0.180,3,2025-05-01


In [ ]:
# Optionally save
neighbors_df.to_csv("england_wales_local_authority_neighbors.csv", index=False)